# 01A — Filtering Variables in a Pediatric no VAD linear EDPVR patient 

In this notebook, we import all the variables identified in "01_Data/Classified_Variables.xlxs' and filter out all the ones that are not used in the specific no VAD, linear EDPVR, simulink simulation of the pediatric patient. To do this, the '04_Matlab_&_Simulink/Simulink_vars_code.m' was ran, which returned '01_Data/simulink_used_variables_no_VAD_linear_EDPVR.csv'. This csv, contains information about each variable and how it was used in the simulink model. From this, we can filter out the non-essential variables. 

### Imports

In [30]:
import pandas as pd 
from pathlib import Path 

### Dataframes 

In [31]:
data_dir = Path("../01_Data")

classified = pd.read_excel(data_dir / "Classified_Variables.xlsx")
used = pd.read_csv(data_dir / "simulink_used_variables_no_VAD_linear_EDPVR.csv")

In [32]:
classified.head()

,Variable,Category,Tunable,Explanation,Equation,Flagged
0,weight,fixed_patient_target,No,Patient body weight,NaN,NaN
1,BSA,fixed_patient_target,No,Body surface area: estimated total surface are...,NaN,NaN
2,hr,fixed_patient_target,No,Heart rate,NaN,NaN
3,SAP,fixed_patient_target,No,Systolic arterial pressure; origin: Gupta,NaN,NaN
4,DAP,fixed_patient_target,No,Diastolic arterial pressure; origin: Gupta,NaN,NaN


In [33]:
used.head()

,Name,Source,SourceType,Users
0,C_can_H,base workspace,base workspace,MyComplexModel_V13R2023a/HM3/outflow cannula1/...
1,Cpulmart,base workspace,base workspace,MyComplexModel_V13R2023a/1//Cpulmart | MyCompl...
2,Cpulmven,base workspace,base workspace,MyComplexModel_V13R2023a/Gain2 | MyComplexMode...
3,Csysart,base workspace,base workspace,MyComplexModel_V13R2023a/1//Cart | MyComplexMo...
4,Csysven,base workspace,base workspace,MyComplexModel_V13R2023a/1//Cven | MyComplexMo...


### Filtering

We clean column names and text values so the Excel classification table and Simulink usage table can be compared reliably. This avoids errors caused by extra spaces, capitalization differences, or missing values.

In [34]:
classified.columns = (
    classified.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

used.columns = (
    used.columns
    .str.strip()
    .str.lower()
)

classified["variable"] = classified["variable"].astype(str).str.strip()
classified["category"] = classified["category"].astype(str).str.strip()
classified["tunable"] = classified["tunable"].astype(str).str.strip()

used["name"] = used["name"].astype(str).str.strip()
used["users"] = used["users"].astype(str)

Direct Usage 

We compare the variables from the classification table with the variables detected in Simulink. Variables found in the Simulink usage file are marked as `directly_used`, and their corresponding Simulink block paths are stored in `simulink_users`.

In [35]:
directly_used_names = set(used["name"].dropna())

classified["simulink_usage"] = classified["variable"].apply(
    lambda x: "directly_used" if x in directly_used_names else "not_directly_used"
)

users_map = (
    used.groupby("name")["users"]
    .apply(lambda x: " | ".join(x.dropna().astype(str)))
    .to_dict()
)

classified["simulink_users"] = classified["variable"].map(users_map)

VAD inactive branch

Since the current case is no-VAD, variables used only in VAD, pump, HM3, or graft blocks are marked as inactive. Obvious pump variables such as `rpm`, `L`, `R`, and `st_VADtype` are also manually marked as inactive.

In [36]:
vad_keywords = ["VAD", "vad", "miniVAD", "HM3", "pump", "Pump", "graft"]

# Fix NaN / float values in users column
used["users"] = used["users"].fillna("").astype(str)

used["is_vad_related"] = used["users"].apply(
    lambda x: any(keyword in x for keyword in vad_keywords)
)

usage_summary = used.groupby("name").agg(
    all_users=("users", lambda x: " | ".join(x.dropna().astype(str))),
    any_vad_related=("is_vad_related", "any"),
    all_vad_related=("is_vad_related", "all"),
).reset_index()

vad_only_vars = set(
    usage_summary.loc[usage_summary["all_vad_related"], "name"]
)

classified["scenario_relevance"] = "active_or_unknown"

classified.loc[
    classified["variable"].isin(vad_only_vars),
    "scenario_relevance"
] = "inactive_VAD_branch"

obvious_vad_vars = [
    "C_can_H",
    "rpm",
    "st_VADtype",
    "st_VAD",
    "L",
    "R",
]

classified.loc[
    classified["variable"].isin(obvious_vad_vars),
    "scenario_relevance"
] = "inactive_VAD_branch"

Linear branch

Because the current setup uses the linear pressure-volume branch, nonlinear EDPVR/ESPVR curve variables are marked as inactive. The `k_Vs_LV_offset` and `k_Vs_RV_offset` tunables are also excluded because they mainly affect the quadratic ESPVR formulation.

In [37]:
nonlinear_edpvr_vars = [
    "V30", "V15", "V30_LV", "V15_LV",
    "V30_RV", "V15_RV",
    "V30_IS",

    "alpha_LV", "beta_LV",
    "alpha_RV", "beta_RV",
    "alpha_IS", "beta_IS",

    "phi_p_LV", "EDPVR_LV",
    "phi_a_LV", "ESPVR_LV",

    "phi_p_RV", "EDPVR_RV",
    "phi_a_RV", "ESPVR_RV",

    "phi_p_IS", "EDP_IS",
    "phi_a_IS", "ESP_IS",

    "Vs_IS", "Ps_IS", "Vd_IS", "V0_IS", "Vm_IS", "Pm_IS",
    "Poffset_IS",

    "k_Vs_IS", "k_Vd_IS", "k_V0_IS", "k_Vm_IS",

    "vol",
]

classified.loc[
    classified["variable"].isin(nonlinear_edpvr_vars),
    "scenario_relevance"
] = "inactive_nonlinear_EDPVR_branch"

#k_Vs_LV_offset and k_Vs_RV_offset mostly affect the quadratic ESPVR through Vs_LV and Vs_RV 

linear_inactive_tunables = [
    "k_Vs_LV_offset",
    "k_Vs_RV_offset",
]

classified.loc[
    classified["variable"].isin(linear_inactive_tunables),
    "scenario_relevance"
] = "inactive_linear_branch"

Users Column

This check shows where selected ventricular variables are used in the Simulink model. It helps verify whether `Vs_LV` and `Vs_RV` are only linked to the nonlinear/quadratic branch, while `phi_a_lin_LV` and `phi_a_lin_RV` belong to the linear branch.

In [38]:
print(used[used["name"].isin(["Vs_LV", "Vs_RV", "phi_a_lin_LV", "phi_a_lin_RV"])][
    ["name", "users"]
])

     name                                              users
95  Vs_LV   MyComplexModel_V13R2023a/Left Ventricle/phi_a/V*
97  Vs_RV  MyComplexModel_V13R2023a/Right Ventricle/phi_a/V*


Indirect Mapping

The `k_...` coefficients are not usually used directly inside Simulink. This mapping links each tunable coefficient to the model variables it affects, so we can keep tunables that indirectly modify active Simulink variables.

In [39]:
indirect_mapping = {
    "k_Vtot": [
        "Vtot", "Vsv_tot", "Vusv_tot",
        "Vusv_sys", "Vusv_pulm",
        "Vusv_sys_ven", "Vusv_pulm_ven"
    ],
    "k_Vsys": [
        "Vsys", "Vpulm", "Vusv_sys",
        "Vusv_pulm", "Vusv_sys_ven", "Vusv_pulm_ven"
    ],
    "k_Vusv_sys": [
        "Vusv_sys", "Vusv_pulm",
        "Vusv_sys_ven", "Vusv_pulm_ven"
    ],
    "k_Vusv_sys_ven": ["Vusv_sys_ven"],
    "k_Vusv_pulm_ven": ["Vusv_pulm_ven"],

    "k_Ctot": [
        "Ctot", "Csys", "Cpulm",
        "Csysven", "Cpulmven", "mcfp"
    ],
    "k_Csys": ["Csys", "Cpulm", "Csysven", "Cpulmven"],

    "k_Rsysven": ["Rsysven", "Rsysart", "mcfp"],
    "k_Rpulmart": ["Rpulmart", "Rpulmven"],

    "k_ESP_LV": ["ESP_LV", "Ps_LV", "phi_a_lin_LV"],
    "k_ESP_RV": ["ESP_RV", "Ps_RV", "phi_a_lin_RV"],
}

Indirect usage

This step checks whether each tunable coefficient affects at least one active Simulink variable. If it does, the coefficient is marked as `indirectly_used` and the affected active variables are stored in `reason_filtering`.

In [40]:
classified["reason_filtering"] = ""

active_direct_vars = set(
    classified.loc[
        (classified["simulink_usage"] == "directly_used") &
        (~classified["scenario_relevance"].str.contains("inactive", na=False)),
        "variable"
    ]
)

for k_var, affected_vars in indirect_mapping.items():
    affected_active_used = [v for v in affected_vars if v in active_direct_vars]

    if affected_active_used:
        is_inactive = classified.loc[
            classified["variable"] == k_var,
            "scenario_relevance"
        ].astype(str).str.contains("inactive", na=False).any()

        if not is_inactive:
            classified.loc[
                classified["variable"] == k_var,
                "simulink_usage"
            ] = "indirectly_used"

            classified.loc[
                classified["variable"] == k_var,
                "reason_filtering"
            ] = "Affects active Simulink variables: " + ", ".join(affected_active_used)

### Choosing what to keep

We first set all variables to `False`, meaning they are not kept by default. The next filtering steps will explicitly mark only the relevant variables as `True`.

In [41]:
classified["keep_for_current_pipeline"] = False

Fixed patient targets 

In [42]:
classified.loc[
    classified["category"] == "fixed_patient_target",
    "keep_for_current_pipeline"
] = True

Active directly/indirectly used variables 

In [43]:
classified.loc[
    classified["simulink_usage"].isin(["directly_used", "indirectly_used"]) &
    (~classified["scenario_relevance"].str.contains("inactive", na=False)),
    "keep_for_current_pipeline"
] = True

Active first-stage tunables 

In [44]:
classified.loc[
    (classified["category"] == "uncertain_coefficient") &
    (classified["tunable"].str.lower() == "yes") &
    (~classified["scenario_relevance"].str.contains("inactive", na=False)),
    "keep_for_current_pipeline"
] = True

Simulation output targets 

In [45]:
classified.loc[
    classified["category"] == "simulation_output_target",
    "keep_for_current_pipeline"
] = True

Model Switches 

In [46]:
classified.loc[
    classified["category"] == "model_switch",
    "keep_for_current_pipeline"
] = True

Linear inactive vars

In [47]:
linear_inactive_vars = [
    "Vs_LV",
    "Vd_LV",
    "Vs_RV",
    "Vd_RV",
    "k_Vs_LV_offset",
    "k_Vs_RV_offset",
]

classified.loc[
    classified["variable"].isin(linear_inactive_vars),
    "scenario_relevance"
] = "inactive_linear_branch"

Filtered Table 

In [48]:
filtered = classified[classified["keep_for_current_pipeline"]].copy()

print("Original count:", len(classified))
print("Filtered count:", len(filtered))

filtered[[
    "variable",
    "category",
    "tunable",
    "simulink_usage",
    "scenario_relevance",
    "keep_for_current_pipeline",
    "reason_filtering"
]].head(50)

Original count: 257
Filtered count: 90


,variable,category,tunable,simulink_usage,scenario_relevance,keep_for_current_pipeline,reason_filtering
0,weight,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
1,BSA,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
2,hr,fixed_patient_target,No,directly_used,active_or_unknown,True,
3,SAP,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
4,DAP,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
6,CVP,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
7,LAP,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
8,sPAP,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
9,dPAP,fixed_patient_target,No,not_directly_used,active_or_unknown,True,
11,EDV_LV,fixed_patient_target,No,not_directly_used,active_or_unknown,True,


### Save outputs

In [49]:
output_dir = data_dir / "processed_pediatric_noVAD_linear"
output_dir.mkdir(parents=True, exist_ok=True)

filtered.to_csv(
    output_dir / "classified_variables_pediatric_dcm_noVAD_linear_filtered.csv",
    index=False
)

filtered[
    (filtered["category"] == "uncertain_coefficient") &
    (filtered["tunable"].str.lower() == "yes")
].to_csv(
    output_dir / "stage_1_tunables_pediatric_noVAD_linear.csv",
    index=False
)

filtered[
    filtered["category"] == "fixed_patient_target"
].to_csv(
    output_dir / "fixed_patient_targets_pediatric_noVAD_linear.csv",
    index=False
)

filtered[
    filtered["category"] == "simulation_output_target"
].to_csv(
    output_dir / "simulation_output_targets_pediatric_noVAD_linear.csv",
    index=False
)